# 🏥 Aged Care Demand Forecasting — Australian Public Sector
## Notebook 07: Executive Report & Interactive Dashboard

> **Audience:** Department of Social Services — Senior Policy Officers & Planning Team
> **Purpose:** Summary findings, actionable recommendations, and interactive dashboard

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import xgboost as xgb
import joblib
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../models')
REPORTS_DIR   = Path('../reports')
APP_DIR       = Path('../app')
APP_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Load all outputs from notebooks 04 and 05
master   = pd.read_csv(PROCESSED_DIR / 'master_with_predictions.csv',
                       dtype={'sa2_code': str})
proj_df  = pd.read_csv(PROCESSED_DIR / 'demand_projections_sa2.csv',
                       dtype={'sa2_code': str})
topic_df = pd.read_csv(PROCESSED_DIR / 'topic_sentiment.csv')
pop_df   = pd.read_csv(RAW_DIR / 'abs_population_projections.csv',
                       dtype={'sa2_code': str})

# Load XGBoost model
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(MODELS_DIR / 'xgb_demand_risk.json')

LABEL_MAP   = {0: 'Low', 1: 'Medium', 2: 'High'}
RISK_COLORS = {'Low': '#55A868', 'Medium': '#DD8452', 'High': '#C44E52'}

# Merge projection data into master where needed
master = master.merge(
    proj_df[['sa2_code', 'proj_recipients_2031', 'proj_recipients_2041',
             'proj_growth_2031', 'proj_growth_2041']],
    on='sa2_code', how='left'
)
master = master.merge(
    pop_df[['sa2_code', 'pop_70plus_2031', 'pop_70plus_2041']],
    on='sa2_code', how='left', suffixes=('', '_p')
)
for col in ['pop_70plus_2031', 'pop_70plus_2041']:
    if col + '_p' in master.columns:
        master[col] = master[col].combine_first(master[col + '_p'])
        master = master.drop(columns=[col + '_p'])

print('✅ All outputs loaded')
print(f'   Master SA2 regions: {len(master):,}')
print(f'   Projection SA2s:    {len(proj_df):,}')
print(f'   Policy documents:   {len(topic_df):,}')

---
## 1. Executive Summary — Key Statistics

In [ ]:
high_risk_n = (master['predicted_risk_label'] == 'High').sum()
med_risk_n  = (master['predicted_risk_label'] == 'Medium').sum()
low_risk_n  = (master['predicted_risk_label'] == 'Low').sum()
total_sa2   = len(master)

pop_70_2024 = master['pop_70plus'].sum()
pop_70_2031 = master['pop_70plus_2031'].sum()
pop_70_2041 = master['pop_70plus_2041'].sum()

curr_recipients  = master['total_recipients'].sum()
proj_2031        = master['proj_recipients_2031'].sum()
proj_2041        = master['proj_recipients_2041'].sum()
growth_pct_2031  = (proj_2031 / curr_recipients - 1) * 100
growth_pct_2041  = (proj_2041 / curr_recipients - 1) * 100

print('=' * 58)
print('  AGED CARE DEMAND FORECASTING — EXECUTIVE SUMMARY')
print('=' * 58)
print()
print('  SA2 REGIONS ANALYSED')
print(f'    Total SA2 regions:        {total_sa2:,}')
print(f'    High demand risk:         {high_risk_n}  ({high_risk_n/total_sa2*100:.0f}%)')
print(f'    Medium demand risk:       {med_risk_n}  ({med_risk_n/total_sa2*100:.0f}%)')
print(f'    Low demand risk:          {low_risk_n}  ({low_risk_n/total_sa2*100:.0f}%)')
print()
print('  POPULATION PROJECTIONS (70+)')
print(f'    2024 baseline:            {pop_70_2024:>10,.0f}')
print(f'    2031 projected:           {pop_70_2031:>10,.0f}  (+{(pop_70_2031/pop_70_2024-1)*100:.1f}%)')
print(f'    2041 projected:           {pop_70_2041:>10,.0f}  (+{(pop_70_2041/pop_70_2024-1)*100:.1f}%)')
print()
print('  DEMAND PROJECTIONS (recipients)')
print(f'    2024/25 current:          {curr_recipients:>10,.0f}')
print(f'    2031 projected:           {proj_2031:>10,.0f}  (+{growth_pct_2031:.1f}%)')
print(f'    2041 projected:           {proj_2041:>10,.0f}  (+{growth_pct_2041:.1f}%)')
print('=' * 58)

---
## 2. Executive Dashboard — Static (Matplotlib)

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: Risk Label Distribution ──────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
risk_counts = master['predicted_risk_label'].value_counts().reindex(['High','Medium','Low'])
bars = ax1.bar(risk_counts.index, risk_counts.values,
               color=[RISK_COLORS[l] for l in risk_counts.index],
               edgecolor='white', width=0.6)
ax1.set_title('Demand Risk Distribution', fontweight='bold')
ax1.set_ylabel('SA2 regions')
for bar, val in zip(bars, risk_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', fontweight='bold')

# ── Panel 2: 70+ Population Projections ───────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
years = ['2024', '2031', '2041']
pops  = [pop_70_2024, pop_70_2031, pop_70_2041]
bars2 = ax2.bar(years, [p/1e6 for p in pops],
                color=['#4C72B0','#DD8452','#C44E52'], edgecolor='white', width=0.5)
ax2.set_title('70+ Population Projections', fontweight='bold')
ax2.set_ylabel('Population (millions)')
for bar, val in zip(bars2, pops):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{val/1e6:.2f}M', ha='center', fontsize=9, fontweight='bold')

# ── Panel 3: Service Gap Distribution ─────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
gap_data = master['service_gap_score'].dropna()
ax3.hist(gap_data, bins=20, color='#4C72B0', edgecolor='white', alpha=0.85)
ax3.axvline(gap_data.mean(), color='red', linestyle='--',
            label=f'Mean={gap_data.mean():.1f}')
ax3.set_title('Service Gap Score Distribution', fontweight='bold')
ax3.set_xlabel('Gap Score (vs national median)')
ax3.set_ylabel('SA2 regions')
ax3.legend(fontsize=9)

# ── Panel 4: Demand Projection by State ───────────────────────────
ax4 = fig.add_subplot(gs[1, :])
state_proj = (
    master[master['state'].notna() & (master['state'] != 'OT')]
    .groupby('state')[['total_recipients','proj_recipients_2031','proj_recipients_2041']]
    .sum()
)
x = np.arange(len(state_proj))
w = 0.25
ax4.bar(x - w, state_proj['total_recipients']/1e3,     width=w, label='2024', color='#4C72B0')
ax4.bar(x,     state_proj['proj_recipients_2031']/1e3, width=w, label='2031', color='#DD8452')
ax4.bar(x + w, state_proj['proj_recipients_2041']/1e3, width=w, label='2041', color='#C44E52')
ax4.set_xticks(x)
ax4.set_xticklabels(state_proj.index)
ax4.set_title('Projected Aged Care Recipients by State (thousands)', fontweight='bold')
ax4.set_ylabel("Recipients ('000s)")
ax4.legend(fontsize=9)

# ── Panel 5: Sentiment Trend ───────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
yearly_sent = topic_df.groupby('year')['vader_compound'].mean()
bar_colors  = ['#C44E52' if v < 0 else '#55A868' for v in yearly_sent]
ax5.bar(yearly_sent.index, yearly_sent.values, color=bar_colors, edgecolor='white', width=0.6)
ax5.axhline(0, color='black', linewidth=0.8)
ax5.set_title('Policy Sentiment Trend', fontweight='bold')
ax5.set_ylabel('VADER Compound Score')
ax5.set_xticks(yearly_sent.index)
ax5.tick_params(axis='x', rotation=45)

# ── Panel 6: Risk by Remoteness ────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1:])
remote_labels = {1:'Major City', 2:'Inner Regional', 3:'Outer Regional',
                 4:'Remote', 5:'Very Remote'}
master['remoteness_label'] = master['remoteness_cat'].map(remote_labels)
remote_risk = (
    master[master['remoteness_label'].notna()]
    .groupby(['remoteness_label','predicted_risk_label']).size()
    .unstack(fill_value=0)
)
for col in ['Low','Medium','High']:
    if col not in remote_risk.columns:
        remote_risk[col] = 0
remote_risk = remote_risk[['Low','Medium','High']]
order = ['Major City','Inner Regional','Outer Regional','Remote','Very Remote']
remote_risk = remote_risk.reindex([o for o in order if o in remote_risk.index])
remote_risk.plot(kind='bar', ax=ax6, stacked=True,
                 color=['#55A868','#DD8452','#C44E52'], edgecolor='white', width=0.7)
ax6.set_title('Risk Distribution by Remoteness', fontweight='bold')
ax6.set_ylabel('SA2 regions')
ax6.set_xlabel('')
ax6.tick_params(axis='x', rotation=20)
ax6.legend(title='Risk', bbox_to_anchor=(1.01, 1))

plt.suptitle('Aged Care Demand Forecasting — Executive Dashboard',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig(REPORTS_DIR / 'executive_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved to reports/executive_dashboard.png')

---
## 3. Interactive Plotly Dashboard

In [ ]:
# Demand projection timeline
years_ts  = ['2024/25', '2031', '2041']
totals_ts = [curr_recipients, proj_2031, proj_2041]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=years_ts, y=totals_ts,
    mode='lines+markers+text',
    text=[f'{v/1e6:.2f}M' for v in totals_ts],
    textposition='top center',
    line=dict(color='#DD8452', width=3),
    marker=dict(size=10, color=['#4C72B0','#DD8452','#C44E52']),
    name='Total Recipients'
))
fig.update_layout(
    title='Aged Care Demand Projection — National (2024→2041)',
    xaxis_title='Year',
    yaxis_title='Total Recipients',
    template='plotly_white',
    yaxis=dict(tickformat=',')
)
fig.write_html(REPORTS_DIR / 'exec_01_demand_projection.html')
fig.show()

# Top 20 high-risk SA2s
top20 = master.nlargest(20, 'prob_high')[[
    'sa2_name', 'state', 'predicted_risk_label', 'prob_high',
    'total_recipients', 'proj_recipients_2031',
    'growth_rate_70plus_2031', 'remoteness_cat', 'irsd_score'
]].reset_index(drop=True)

fig2 = px.bar(
    top20, x='prob_high', y='sa2_name', orientation='h',
    color='prob_high', color_continuous_scale='Reds',
    labels={'prob_high': 'P(High Risk)', 'sa2_name': 'SA2 Region'},
    title='Top 20 SA2 Regions — Probability of High Demand Risk'
)
fig2.update_layout(template='plotly_white', yaxis={'categoryorder':'total ascending'})
fig2.write_html(REPORTS_DIR / 'exec_02_top20_risk.html')
fig2.show()

---
## 4. Policy Recommendations

In [ ]:
print('=' * 65)
print('  POLICY RECOMMENDATIONS')
print('=' * 65)

recommendations = [
    {
        'priority': 'IMMEDIATE (0–2 years)',
        'items': [
            'Target emergency Home Care Package allocations to top 10 high-risk SA2 regions',
            'Fast-track service expansion in Very Remote (cat 5) SA2s with high growth rates',
            'Workforce recruitment campaigns in Remote and Outer Regional areas',
        ]
    },
    {
        'priority': 'MEDIUM TERM (2–5 years)',
        'items': [
            'Commission infrastructure planning studies in all High-risk SA2 regions',
            'Expand telehealth and technology-enabled care in regions with low utilisation',
            'Pilot integrated aged care + primary health hubs in Remote SA2s',
        ]
    },
    {
        'priority': 'LONG TERM (5–10 years)',
        'items': [
            'Refresh projections annually as ABS releases updated population estimates',
            'Expand model to include NDIS co-design for dual eligible cohorts',
            'Integrate cost modelling to quantify funding gaps alongside demand gaps',
            'Replace SA3-level AIHW data with SA2-level data when GEN releases it',
        ]
    },
]

for rec in recommendations:
    print(f"\n  {rec['priority']}")
    for item in rec['items']:
        print(f'    • {item}')
print()
print('=' * 65)

---
## 5. Model Card

In [ ]:
model_card = """
╔══════════════════════════════════════════════════════════════════╗
║           MODEL CARD — AGED CARE DEMAND RISK CLASSIFIER          ║
╠══════════════════════════════════════════════════════════════════╣
║ Model name:     XGBoost Demand Risk Classifier (v1.0)            ║
║ Model type:     Gradient Boosted Trees (XGBoost)                 ║
║ Task:           Multiclass classification (Low/Medium/High)      ║
║ Geography:      SA2 regions, Australia (~2,454 regions)          ║
║ Date trained:   2026                                             ║
╠══════════════════════════════════════════════════════════════════╣
║ TRAINING DATA                                                    ║
║   ABS SEIFA 2021 (IRSD, IRSAD)                                  ║
║   ABS Regional Population by Age & Sex 2024                     ║
║   ABS Population Projections Series B 2031/2041                 ║
║   AIHW GEN Aged Care Recipients 2024-25 (SA3→SA2 distributed)  ║
║   Level: SA2 aggregate — no individual records                  ║
║   Licence: CC BY 4.0 (all sources)                              ║
╠══════════════════════════════════════════════════════════════════╣
║ TARGET VARIABLE                                                  ║
║   demand_risk_label: High / Medium / Low                        ║
║   Derived from: growth rate × service gap × remoteness          ║
║   Thresholds: percentile-based (75th pct for High)             ║
╠══════════════════════════════════════════════════════════════════╣
║ PERFORMANCE                                                      ║
║   CV Accuracy:     See Notebook 05 output                       ║
║   Macro AUC:       See Notebook 05 output                       ║
║   Explainability:  SHAP TreeExplainer (Notebook 05)            ║
╠══════════════════════════════════════════════════════════════════╣
║ INTENDED USE                                                     ║
║   ✅  Regional infrastructure and workforce planning            ║
║   ✅  Budget allocation prioritisation across SA2 regions       ║
║   ❌  Individual care eligibility assessments                   ║
║   ❌  Provider performance benchmarking                         ║
╠══════════════════════════════════════════════════════════════════╣
║ ETHICAL CONSIDERATIONS                                           ║
║   • Remote/Indigenous communities may be underserved;           ║
║     high risk scores should trigger MORE investment, not less   ║
║   • IRSD is structural — disadvantage is systemic               ║
║   • Projections assume ABS Series B (medium growth)             ║
║   • AIHW data distributed SA3→SA2 using pop_70plus weighting;  ║
║     SA2-level precision limited by this disaggregation          ║
╠══════════════════════════════════════════════════════════════════╣
║ LIMITATIONS                                                      ║
║   • Supply features proxy from utilisation — no raw supply data ║
║   • No quarterly time series — snapshot data only               ║
║   • SA2-level variation within SA3s estimated, not observed     ║
║   • Does not model workforce supply constraints                 ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(model_card)
with open(REPORTS_DIR / 'model_card.txt', 'w') as f:
    f.write(model_card)
print('✅ Model card saved to reports/model_card.txt')

---
## 6. Plotly Dash Interactive Dashboard App

In [ ]:
dashboard_code = '''"""
Aged Care Demand Forecasting — Plotly Dash Dashboard
Run with: python app/dashboard.py
Then open: http://localhost:8050
"""
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, dash_table
import dash_bootstrap_components as dbc
from pathlib import Path

BASE  = Path(__file__).parent.parent
PROC  = BASE / "data" / "processed"
RAW   = BASE / "data" / "raw"

master   = pd.read_csv(PROC / "master_with_predictions.csv", dtype={"sa2_code": str})
proj_df  = pd.read_csv(PROC / "demand_projections_sa2.csv",  dtype={"sa2_code": str})
topic_df = pd.read_csv(PROC / "topic_sentiment.csv")
pop_df   = pd.read_csv(RAW  / "abs_population_projections.csv", dtype={"sa2_code": str})

master = master.merge(
    proj_df[["sa2_code","proj_recipients_2031","proj_recipients_2041","proj_growth_2031"]],
    on="sa2_code", how="left"
)
master = master.merge(
    pop_df[["sa2_code","pop_70plus_2031","pop_70plus_2041"]],
    on="sa2_code", how="left", suffixes=("","_p")
)
for col in ["pop_70plus_2031","pop_70plus_2041"]:
    if col+"_p" in master.columns:
        master[col] = master[col].combine_first(master[col+"_p"])
        master = master.drop(columns=[col+"_p"])

RISK_COLORS = {"Low": "#55A868", "Medium": "#DD8452", "High": "#C44E52"}
STATES = sorted([s for s in master["state"].dropna().unique() if s != "OT"])

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY],
           title="Aged Care Demand Forecasting")

app.layout = dbc.Container([
    dbc.Row(dbc.Col(html.Div([
        html.H2("🏥 Aged Care Demand Forecasting Dashboard", className="text-white mb-1"),
        html.P("Australian Public Sector | SA2 Regional Analysis | 2026",
               className="text-white-50 mb-0")
    ], className="p-3", style={"background":"#2C3E50","borderRadius":"8px"}))),
    html.Hr(),
    dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H4(f"{len(master):,}", className="card-title text-primary"),
            html.P("SA2 Regions Analysed", className="card-text text-muted")
        ]), className="shadow-sm text-center"), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H4(str((master["predicted_risk_label"]=="High").sum()),
                    className="card-title text-danger"),
            html.P("High Risk SA2s", className="card-text text-muted")
        ]), className="shadow-sm text-center"), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H4(f"{master['pop_70plus_2031'].sum():,.0f}",
                    className="card-title text-warning"),
            html.P("Projected 70+ Pop (2031)", className="card-text text-muted")
        ]), className="shadow-sm text-center"), width=3),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H4(f"{master['proj_recipients_2031'].sum():,.0f}",
                    className="card-title text-info"),
            html.P("Projected Recipients (2031)", className="card-text text-muted")
        ]), className="shadow-sm text-center"), width=3),
    ], className="my-3"),
    html.Hr(),
    dbc.Row([
        dbc.Col([
            html.Label("Filter by State:"),
            dcc.Dropdown(id="state-filter",
                options=[{"label": s, "value": s} for s in STATES],
                value=None, placeholder="All states", clearable=True)
        ], width=4),
        dbc.Col([
            html.Label("Filter by Risk Level:"),
            dcc.Dropdown(id="risk-filter",
                options=[{"label": r, "value": r} for r in ["High","Medium","Low"]],
                value=None, placeholder="All risk levels", clearable=True)
        ], width=4),
    ], className="mb-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(id="risk-bar"), width=6),
        dbc.Col(dcc.Graph(id="projection-bar"), width=6),
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id="scatter-gap"), width=6),
        dbc.Col(dcc.Graph(id="sentiment-bar"), width=6),
    ]),
    html.Hr(),
    html.H5("SA2 Region Detail Table", className="mt-3"),
    dash_table.DataTable(
        id="sa2-table",
        columns=[
            {"name": "SA2 Name",       "id": "sa2_name"},
            {"name": "State",          "id": "state"},
            {"name": "Risk Label",     "id": "predicted_risk_label"},
            {"name": "P(High)",        "id": "prob_high",             "type": "numeric", "format": {"specifier": ".3f"}},
            {"name": "Recipients",     "id": "total_recipients",      "type": "numeric", "format": {"specifier": ","}},
            {"name": "Proj 2031",      "id": "proj_recipients_2031",  "type": "numeric", "format": {"specifier": ","}},
            {"name": "Growth Rate",    "id": "growth_rate_70plus_2031","type":"numeric", "format": {"specifier": ".2%"}},
            {"name": "IRSD Score",     "id": "irsd_score",            "type": "numeric", "format": {"specifier": ".0f"}},
            {"name": "Remoteness",     "id": "remoteness_cat",        "type": "numeric"},
        ],
        page_size=15,
        sort_action="native",
        filter_action="native",
        style_table={"overflowX": "auto"},
        style_cell={"fontSize": "12px", "padding": "4px"},
        style_data_conditional=[
            {"if": {"filter_query": '{predicted_risk_label} = "High"'},
             "backgroundColor": "#fdecea", "color": "#C44E52"},
            {"if": {"filter_query": '{predicted_risk_label} = "Medium"'},
             "backgroundColor": "#fff3e0"},
        ],
    ),
], fluid=True)

def filter_data(state, risk):
    df = master.copy()
    if state: df = df[df["state"] == state]
    if risk:  df = df[df["predicted_risk_label"] == risk]
    return df

@app.callback(
    Output("risk-bar","figure"), Output("projection-bar","figure"),
    Output("scatter-gap","figure"), Output("sa2-table","data"),
    Input("state-filter","value"), Input("risk-filter","value")
)
def update_all(state, risk):
    df = filter_data(state, risk)

    # Risk distribution
    rc = df["predicted_risk_label"].value_counts().reindex(["High","Medium","Low"]).fillna(0)
    fig1 = go.Figure(go.Bar(
        x=rc.index, y=rc.values,
        marker_color=[RISK_COLORS[l] for l in rc.index],
        text=rc.values, textposition="outside"
    ))
    fig1.update_layout(title="Demand Risk Distribution", template="plotly_white",
                       showlegend=False, yaxis_title="SA2 count")

    # Projection bars
    years   = ["2024/25","2031","2041"]
    totals  = [df["total_recipients"].sum(),
               df["proj_recipients_2031"].sum(),
               df["proj_recipients_2041"].sum()]
    fig2 = go.Figure(go.Bar(
        x=years, y=totals,
        marker_color=["#4C72B0","#DD8452","#C44E52"],
        text=[f"{v:,.0f}" for v in totals], textposition="outside"
    ))
    fig2.update_layout(title="Recipient Demand Projection", template="plotly_white",
                       yaxis_title="Total recipients")

    # Scatter: service gap vs growth rate
    fig3 = px.scatter(
        df.dropna(subset=["service_gap_score","growth_rate_70plus_2031"]),
        x="service_gap_score", y="growth_rate_70plus_2031",
        color="predicted_risk_label", color_discrete_map=RISK_COLORS,
        hover_name="sa2_name", size="total_recipients",
        size_max=20, opacity=0.7,
        labels={"service_gap_score":"Service Gap Score",
                "growth_rate_70plus_2031":"70+ Growth Rate 2024-2031"},
        title="Service Gap vs Population Growth"
    )
    fig3.update_layout(template="plotly_white")

    tbl_cols = ["sa2_name","state","predicted_risk_label","prob_high",
                "total_recipients","proj_recipients_2031",
                "growth_rate_70plus_2031","irsd_score","remoteness_cat"]
    tbl_data = df[tbl_cols].fillna("").to_dict("records")

    return fig1, fig2, fig3, tbl_data

@app.callback(
    Output("sentiment-bar","figure"),
    Input("state-filter","value")
)
def update_sentiment(_):
    ys = topic_df.groupby("year")["vader_compound"].mean().reset_index()
    fig = go.Figure(go.Bar(
        x=ys["year"], y=ys["vader_compound"],
        marker_color=["#C44E52" if v < 0 else "#55A868" for v in ys["vader_compound"]]
    ))
    fig.update_layout(title="Policy Sentiment Trend (VADER)",
                      template="plotly_white",
                      xaxis_title="Year", yaxis_title="Compound Score")
    return fig

if __name__ == "__main__":
    app.run(debug=True, port=8050)
'''

# Write dashboard app
with open(APP_DIR / 'dashboard.py', 'w') as f:
    f.write(dashboard_code)
print(f'✅ Dashboard app written to: {APP_DIR / "dashboard.py"}')
print()
print('To run the dashboard:')
print('  cd app')
print('  pip install dash dash-bootstrap-components')
print('  python dashboard.py')
print('  Open: http://localhost:8050')

---
## 7. Project Deliverables Summary

In [ ]:
print('=' * 62)
print('  PROJECT DELIVERABLES')
print('=' * 62)

deliverables = {
    'Notebooks': [
        '01_problem_framing.ipynb      — Research questions & ethics',
        '02_data_collection.ipynb      — ABS + AIHW ingestion pipelines',
        '03_eda.ipynb                  — Exploratory analysis & charts',
        '04_feature_engineering.ipynb  — Feature construction & target labelling',
        '05_modelling.ipynb            — XGBoost + demand projections',
        '06_nlp_policy_analysis.ipynb  — LDA + VADER on policy corpus',
        '07_executive_report.ipynb     — This notebook',
    ],
    'Data (raw)': [
        'seifa_2021.csv                — SEIFA + remoteness (SA2)',
        'abs_population_sa2.xlsx       — ABS population by age',
        'abs_population_projections.csv — Population projections (SA2)',
        'aihw_recipients_sa3.csv       — AIHW recipients (SA3)',
        'aihw_recipients_sa2.csv       — AIHW recipients (SA2, distributed)',
    ],
    'Data (processed)': [
        'master_features.csv           — Full feature set (SA2)',
        'master_with_predictions.csv   — Features + XGBoost predictions',
        'demand_projections_sa2.csv    — 2031/2041 demand projections',
        'X_train/test, y_train/test    — ML train/test splits',
        'topic_sentiment.csv           — LDA + VADER outputs',
    ],
    'Models': [
        'models/xgb_demand_risk.json   — XGBoost classifier',
        'models/shap_explainer.pkl     — SHAP TreeExplainer',
        'models/lda_model.gensim       — LDA topic model',
    ],
    'Reports': [
        'reports/executive_dashboard.png   — Static summary dashboard',
        'reports/model_card.txt            — Model card (ethics & limitations)',
        'reports/exec_01_demand_projection.html',
        'reports/exec_02_top20_risk.html',
    ],
    'App': [
        'app/dashboard.py  — Plotly Dash interactive dashboard',
    ]
}

for section, items in deliverables.items():
    print(f'  {section}:')
    for item in items:
        print(f'    • {item}')
    print()

print('=' * 62)
print('  ✅ PROJECT COMPLETE')
print('=' * 62)